In [5]:
%pip install pandas openpyxl ipykernel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import glob
import pandas as pd

In [7]:
zip_files = sorted(glob.glob("raw/2025*-divvy-tripdata.zip"))
monthly_summaries = []

In [8]:
for file in zip_files:
    df = pd.read_csv(file)
    
    # Convert timestamp to a plain calendar date (YYYY-MM-DD)
    df["date"] = pd.to_datetime(df["started_at"]).dt.floor("D")
    
    # Summarize this month into 1 row per day
    daily = df.groupby("date").agg(
        total_rides=("ride_id", "count"),
        member_rides=("member_casual", lambda s: (s == "member").sum()),
        casual_rides=("member_casual", lambda s: (s == "casual").sum()),
        electric_rides=("rideable_type", lambda s: (s == "electric_bike").sum()),
        classic_rides=("rideable_type", lambda s: (s == "classic_bike").sum())
    ).reset_index()
    
    monthly_summaries.append(daily)

# Stack all 12 monthly summaries into one 365-row table
daily_divvy = (
    pd.concat(monthly_summaries, ignore_index=True)
    .groupby("date", as_index=False)
    .sum()
)

print(daily_divvy.shape)  # Should print (365, 6)
print(daily_divvy.head())

(366, 6)
        date  total_rides  member_rides  casual_rides  electric_rides  \
0 2024-12-31           53            16            37              32   
1 2025-01-01         3562          2321          1241            2280   
2 2025-01-02         4749          3807           942            2933   
3 2025-01-03         4200          3436           764            2690   
4 2025-01-04         3198          2479           719            1966   

   classic_rides  
0             21  
1           1282  
2           1816  
3           1510  
4           1232  


In [10]:
# Keep only 2025 dates (removes the 53 spillover rides from 2024-12-31)
daily_divvy = daily_divvy[daily_divvy["date"] >= "2025-01-01"].reset_index(drop=True)
print("Clean 2025 Divvy shape:", daily_divvy.shape)  # Should be (365, 6)

# Load and inspect the weather spreadsheet
weather = pd.read_excel("raw/Chicago-Weather-2025.xlsx", skiprows=3)
print("Weather shape:", weather.shape)
print("Weather columns:", weather.columns.tolist())
print(weather.head(3))

Clean 2025 Divvy shape: (365, 6)
Weather shape: (365, 8)
Weather columns: ['time', 'temperature_2m_max (Â°F)', 'temperature_2m_min (Â°F)', 'precipitation_sum (inch)', 'snowfall_sum (inch)', 'rain_sum (inch)', 'temperature_2m_mean (Â°F)', 'wind_speed_10m_max (mp/h)']
        time  temperature_2m_max (Â°F)  temperature_2m_min (Â°F)  \
0 2025-01-01                      33.4                      25.5   
1 2025-01-02                      31.5                      24.5   
2 2025-01-03                      26.0                      17.9   

   precipitation_sum (inch)  snowfall_sum (inch)  rain_sum (inch)  \
0                     0.004                0.028              0.0   
1                     0.008                0.055              0.0   
2                     0.000                0.000              0.0   

   temperature_2m_mean (Â°F)  wind_speed_10m_max (mp/h)  
0                       29.7                       13.8  
1                       27.7                       10.8  
2        

In [ ]:
# 1. Fix the 'Â°F' encoding artifact and rename 'time' to 'date'
weather.columns = [c.replace("Â°", "°") for c in weather.columns]
weather["date"] = pd.to_datetime(weather["time"]).dt.floor("D")
weather = weather.drop(columns=["time"]).drop_duplicates(subset=["date"])

# 2. Merge daily ridership and weather on 'date'
merged_df = pd.merge(daily_divvy, weather, on="date", how="left")

print("Merged shape:", merged_df.shape)  # Should print (365, 13)
print("Missing values:\n", merged_df.isna().sum())
print(merged_df.head())

# 3. Export the finished master dataset
merged_df.to_csv("daily_ridership_weather_2025.csv", index=False)